# Phase 1 — Baseline Qwen3.5-9B trên 4 datasets × {zero-shot, few-shot, symbolic-cot}

Notebook này tổ chức mỗi experiment trong **1 cell riêng** để dễ debug: khi 1 experiment lỗi/lạ, bạn chỉ cần chỉnh & rerun cell đó — không ảnh hưởng các experiment khác.

Model Qwen3.5-9B chỉ load **một lần** ở cell setup rồi tái sử dụng cho cả 12 experiment → tiết kiệm VRAM load time.

Mỗi cell experiment bật **verbose** để in:
- Full log 5 sample đầu (câu hỏi, raw output, extracted, gold, correct, thời gian).
- Log rút gọn mỗi N sample (xem `verbose_every`).
- Running metric (F1 / accuracy) mỗi 100 sample để phát hiện sớm prompt kém hoặc extractor sai.

**Symbolic CoT** (EXP 9–12): LLM sinh Python datetime program → symbolic executor chạy deterministic → verify kết quả → self-correct nếu sai. Cấu hình qua `n_hypotheses` và `max_correction_attempts` trong `run_exp()` hoặc trực tiếp trong YAML config.

## Setup (chạy tuần tự 1 lần)

In [ ]:
# === SETUP 1 — cài môi trường ===
!pip install -q -U transformers accelerate scikit-learn pyyaml
# Nếu cần bản dev: !pip install -q -U git+https://github.com/huggingface/transformers.git
# ⚠️ Sau khi chạy cell này: Runtime → Restart session rồi chạy tiếp.

In [ ]:
# === SETUP 2 — mount Drive + clone repo ===
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: đổi
REPO_DIR = '/content/Temporal_Reasoning'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
# === SETUP 3 — cấu hình path + link Dataset từ Drive ===
import os
DRIVE_OUT = '/content/drive/MyDrive/Temporal_Reasoning/outputs'   # nơi lưu predictions + metrics
DATASET_ROOT = '/content/drive/MyDrive/Temporal_Reasoning/Dataset'  # đã upload Dataset/Raw ở đây
os.makedirs(DRIVE_OUT, exist_ok=True)

local_ds = os.path.join(REPO_DIR, 'Dataset')
if not os.path.exists(local_ds):
    os.symlink(DATASET_ROOT, local_ds)
print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
print('DRIVE_OUT:', DRIVE_OUT)

In [ ]:
# === SETUP 4 — preprocess raw → JSONL (Dataset/Preprocessed/) ===
!python -m src.data.preprocess

In [ ]:
# === SETUP 5 — load Qwen3.5-9B một lần, dùng chung cho 12 experiment ===
from src.models.qwen import QwenChatLM, QwenConfig
from src.runner import load_config, run

MODEL = QwenChatLM(QwenConfig(model_name='Qwen/Qwen3.5-9B', dtype='bfloat16'))
MODEL.load()
print('Model ready:', MODEL.config.model_name)

def run_exp(cfg_path, *, verbose=True, verbose_first_n=5, verbose_every=200,
            running_score_every=100, output_dir=DRIVE_OUT,
            n_hypotheses=None, max_correction_attempts=None):
    """Helper: load config, override verbose + output_dir + symbolic_cot params, chạy với MODEL đã load."""
    cfg = load_config(cfg_path)
    cfg.output_dir = output_dir
    cfg.verbose = verbose
    cfg.verbose_first_n = verbose_first_n
    cfg.verbose_every = verbose_every
    cfg.running_score_every = running_score_every
    if n_hypotheses is not None:
        cfg.n_hypotheses = n_hypotheses
    if max_correction_attempts is not None:
        cfg.max_correction_attempts = max_correction_attempts
    return run(cfg, model=MODEL)

## 12 experiments — mỗi experiment 1 cell riêng

Các cell dưới đây độc lập: rerun cell nào chỉ ảnh hưởng kết quả của experiment đó. Output của mỗi experiment lưu tại `DRIVE_OUT/<method>/<dataset>/` (predictions.jsonl + metrics.json) và append 1 dòng vào `DRIVE_OUT/summary.csv`.

### Zero-shot

In [ ]:
# === EXP 1/8 — zero_shot × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/zero_shot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 2/8 — zero_shot × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/zero_shot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 3/8 — zero_shot × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/zero_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 4/8 — zero_shot × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/zero_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### Few-shot (k cố định, shots thủ công trong `src/prompts/shot_pools.py`)

In [ ]:
# === EXP 5/8 — few_shot k=4 × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/few_shot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 6/8 — few_shot k=3 × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/few_shot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 7/8 — few_shot k=3 × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 8/8 — few_shot k=4 × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### Symbolic CoT (program synthesis + symbolic execution)

LLM đóng vai **planner/compiler**: sinh Python datetime program → `temporal_executor` chạy deterministic → verify kết quả → self-correct nếu execution fail.

Tuỳ chỉnh:
- `n_hypotheses`: số chương trình độc lập mỗi sample (1 = nhanh, 3 = robust hơn).
- `max_correction_attempts`: số lần self-correction loop nếu program fail (0 = không sửa).

Các tham số này có thể override trong `run_exp()` hoặc chỉnh trực tiếp trong YAML config.

In [ ]:
# === EXP 9/12 — symbolic_cot × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/symbolic_cot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 10/12 — symbolic_cot × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/symbolic_cot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 11/12 — symbolic_cot × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/symbolic_cot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 12/12 — symbolic_cot × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/symbolic_cot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

## Debug helpers

Các cell dưới để audit chất lượng extractor / xem sample thất bại — dùng khi cần thôi.

In [ ]:
# === DEBUG A — load lại predictions của 1 run, xem các sample sai đầu tiên ===
import json
from pathlib import Path

METHOD = 'zero_shot'        # đổi theo experiment cần audit
DATASET = 'vlsp_date'       # đổi theo experiment cần audit
N_SHOW = 10

path = Path(DRIVE_OUT) / METHOD / DATASET / 'predictions.jsonl'
wrong = []
parse_fail = []
with open(path, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r['extracted'] is None:
            parse_fail.append(r)
        elif not r['correct']:
            wrong.append(r)

print(f'parse_fail={len(parse_fail)} wrong={len(wrong)}')
print('\n--- Sample parse failures ---')
for r in parse_fail[:N_SHOW]:
    print(f"  gold={r['gold_normalized']!r} raw={r['raw_output']!r}")
print('\n--- Sample wrong (but parsed) ---')
for r in wrong[:N_SHOW]:
    print(f"  gold={r['gold_normalized']!r} pred={r['extracted']!r} raw={r['raw_output'][:120]!r}")

In [ ]:
# === DEBUG B — thử prompt 1 sample bất kỳ cho nhanh (không qua runner) ===
from src.data.registry import load_dataset
from src.prompts.templates import build_messages
from src.prompts.shot_pools import get_shots
from src.evaluation.extractor import extract, normalize_gold

DATASET = 'vlsp_date'       # đổi dataset để probe
IDX = 0
K_SHOT = 0                  # 0 = zero-shot

samples = load_dataset(DATASET, max_samples=IDX + 1)
s = samples[IDX]
shots = get_shots(s['task'], s['language'], K_SHOT) if K_SHOT else ()
msgs = build_messages(s, shots=shots)
print('--- MESSAGES ---')
for m in msgs:
    print(f"[{m.role}] {m.content}")

raw = MODEL.generate(msgs, max_new_tokens=32, do_sample=False, enable_thinking=False)
print('\n--- RAW OUTPUT ---')
print(repr(raw))
print('\n--- EXTRACTED vs GOLD ---')
print('extracted:', extract(s['task'], s['language'], raw))
print('gold     :', normalize_gold(s['task'], s['language'], s['gold']))

In [ ]:
# === DEBUG C — xem bảng tổng quan summary.csv ===
import pandas as pd, os
df = pd.read_csv(os.path.join(DRIVE_OUT, 'summary.csv'))
df